# Prototype Idea

## Interactive Insurance Cost Prediction Tool

### Purpose

The tool helps clients and advisers explore illustrative insurance-charge estimates and compare how changing profile details relates to predicted costs. Estimates are educational and are not insurance quotes.


## Analysis insight

The project analysis found that customer details such as smoking status, age, and BMI can help explain differences in insurance charges. The Random Forest regression model implemented below outperformed a median-only baseline on the held-out test data (MAE about $2,456 versus $9,294). The model is exploratory and its estimates are not guaranteed to be accurate for an individual.

## Proposed tool

Build a simple **Insurance Cost Explorer** for clients and insurance advisers. It would let a user enter age, BMI, number of children, smoking status, sex, and region, then show an estimated annual charge and explain that it is a model-based estimate rather than a quote.

## Low-fidelity screen plan

1. **User details:** numeric fields for age, BMI, and children; dropdowns for smoking status, sex, and region.
2. **Estimate action:** a clearly labelled button to calculate an estimate from the trained regression model.
3. **Results:** show the estimated charge, a broad range based on validation errors, and a short plain-language explanation of the entered factors.
4. **Scenario comparison:** allow the user to change one input at a time, such as smoking status, and compare the resulting estimates side by side.
5. **Limitations notice:** state that the dataset is small and may not represent all customers; results are for exploration only and must not be used for underwriting or as medical advice.

## How the prototype helps

Clients can explore how example profiles relate to historical charges and prepare questions for an adviser. Advisers can use scenario comparisons to explain patterns without presenting them as causal or definitive. A Streamlit interface would be suitable because it can provide interactive inputs and results in a lightweight web app.

## Validation before use

The prototype should be checked against a held-out test set and compared with a simple baseline using MAE, RMSE, and R-squared. Before any real client use, it would need more representative data, fairness and privacy reviews, documented model limitations, and review by qualified insurance professionals. The current model is an educational prototype, not a production pricing tool.

## Train and evaluate the insurance-cost model

Because insurance charges are continuous numeric values, this notebook uses regression. It compares a Scikit-learn Linear Regression model and Random Forest regressor using the raw insurance data. Categorical inputs are one-hot encoded inside a pipeline so preprocessing is learned from the training set and remains consistent when making predictions. Both models are evaluated against a median-only baseline on held-out data using MAE, RMSE, and R-squared.

Run the cells below from the project root or from the `jupyter_notebooks` folder. The estimate is for educational exploration only, not an insurance quote, underwriting decision, or medical advice.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

data_candidates = [
    Path.cwd() / "raw_data" / "insurance.csv",
    Path.cwd().parent / "raw_data" / "insurance.csv",
]
data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find raw_data/insurance.csv. Run this notebook from the "
        "project root or the jupyter_notebooks folder."
    )

df = pd.read_csv(data_path)
df.columns = df.columns.str.strip().str.lower()
df = df.drop_duplicates().reset_index(drop=True)

target = "charges"
numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]
feature_columns = numeric_features + categorical_features
required_columns = feature_columns + [target]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"The dataset is missing required columns: {missing_columns}")

df = df.dropna(subset=[target]).copy()
for column in categorical_features:
    df[column] = df[column].astype("string").str.strip().str.lower()

X = df[feature_columns]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
linear_model.fit(X_train, y_train)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
linear_predictions = linear_model.predict(X_test)
baseline_predictions = baseline.predict(X_test)
results = pd.DataFrame(
    {
        "Model": ["Linear Regression", "Random Forest", "Median baseline"],
        "MAE ($)": [
            mean_absolute_error(y_test, linear_predictions),
            mean_absolute_error(y_test, predictions),
            mean_absolute_error(y_test, baseline_predictions),
        ],
        "RMSE ($)": [
            np.sqrt(mean_squared_error(y_test, linear_predictions)),
            np.sqrt(mean_squared_error(y_test, predictions)),
            np.sqrt(mean_squared_error(y_test, baseline_predictions)),
        ],
        "R-squared": [
            r2_score(y_test, linear_predictions),
            r2_score(y_test, predictions),
            r2_score(y_test, baseline_predictions),
        ],
    }
).set_index("Model")

print(f"Loaded {len(df):,} unique records from {data_path}")
print(f"Training records: {len(X_train):,} | Test records: {len(X_test):,}")
results.round(2)


Loaded 1,337 unique records from c:\Users\amissetu\Documents\GitHub\Unit_2_ Summative_ Assessment\Unit-_2-_Summative-_Assessment-_Project\raw_data\insurance.csv
Training records: 1,069 | Test records: 268


,MAE ($),RMSE ($),R-squared
Model,,,
Linear Regression,4177.05,5956.34,0.81
Random Forest,2455.96,4452.27,0.89
Median baseline,9293.84,14442.13,-0.14


In [4]:
# Example profile; edit these values to explore another scenario.
sample_profile = pd.DataFrame(
    [
        {
            "age": 35,
            "bmi": 27.5,
            "children": 1,
            "sex": "female",
            "smoker": "no",
            "region": "northwest",
        }
    ]
)
estimated_charge = model.predict(sample_profile)[0]
test_mae = results.loc["Random Forest", "MAE ($)"]

print(f"Illustrative annual charge estimate: ${estimated_charge:,.0f}")
print(
    "Approximate error band (estimate +/- test MAE): "
    f"${max(0, estimated_charge - test_mae):,.0f} to "
    f"${estimated_charge + test_mae:,.0f}"
)
print("This error band is illustrative, not a statistical prediction interval or quote.")

feature_names = model.named_steps["preprocessor"].get_feature_names_out()
feature_importance = pd.Series(
    model.named_steps["regressor"].feature_importances_, index=feature_names
).sort_values(ascending=False)
feature_importance.head(10).rename("Importance").to_frame()


Illustrative annual charge estimate: $5,330
Approximate error band (estimate +/- test MAE): $2,874 to $7,786
This error band is illustrative, not a statistical prediction interval or quote.


,Importance
categorical__smoker_no,0.478005
numeric__bmi,0.199220
categorical__smoker_yes,0.147423
numeric__age,0.136988
numeric__children,0.018544
categorical__region_northwest,0.004354
categorical__region_northeast,0.004222
categorical__region_southeast,0.003132
categorical__region_southwest,0.002904
categorical__sex_male,0.002705
